In [ ]:
import functools
import numpy as np
import qiskit as qk
import qiskit_aer
import qiskit.providers.fake_provider as qk_fp
import qiskit_dynamics as qk_d
from scipy.integrate import simpson
from scipy.linalg import sqrtm, expm
import qutip as qt
from tqdm import tqdm

import sys

sys.path.append("../")
import pulse_simulator as ps

import matplotlib.pyplot as plt

# Backend

In [ ]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

backend = qk_fp.FakeManila()
dt = backend.configuration().dt * ns
desired_duration = 225  # dt

# Gate time in SI units (granularity restriction)
duration = ps.qiskit_closest_multiple_of_16(desired_duration * dt / ns, backend)

# Load gatesets

In [ ]:
# Load shapes
fname = "../pico-pulses/saved-pulses-2024-01-24/a_single_qubit_gateset_R1e-3.csv"
knames = ["X1", "X2", "SX1", "SX2"]
pico_a = {k: v for k, v in zip(knames, ps.from_csv(fname))}

fig, ax = plt.subplots(figsize=[5, 2])
for k, v in pico_a.items():
    ax.plot(v, label=k)
ax.legend()

In [ ]:
# The initial run takes awhile to construct the symbolic pulse shapes
x_gates = {}
for k, v in pico_a.items():
    x_gates[k] = ps.scalable_symbolic_pulse(
        ps.custom_envelope(duration, v), f"custom-{k}"
    )

In [ ]:
y_gates = {}
for k, v in pico_a.items():
    y_gates[k] = ps.scalable_symbolic_pulse(
        ps.custom_envelope(duration, v), f"custom-{k}-π/2", angle=np.pi / 2
    )

# y_gates["X1"].draw()

In [ ]:
default_gates = {}
for k in pico_a.keys():
    if "SX" in k:
        default_gates[k] = ps.gaussian_envelope(duration, dt=dt, angle=np.pi / 4)
    else:
        default_gates[k] = ps.gaussian_envelope(duration, dt=dt, angle=np.pi / 2)

# default_gates["X1"].draw()

# Build the model

In [ ]:
registers = [0, 1]

# NOTE: If the Rabi rates are True, you have to calibrate!
cvars = ps.backend_simulation_vars(backend, rabi=False, units=Units)

# Carrier frequencies of each control line
carriers = ps.backend_carriers(backend, cvars)

# Partially compile model functions with parameters
# =====
H_rx = functools.partial(
    ps.rx_model,
    registers=registers,
    backend=backend,
    variables=cvars,
    rotating_frame=False,
)

A_decay = functools.partial(ps.qubit_decay_model, registers=registers, variables=cvars)

H_xtalk = ps.crosstalk_model(registers, ps.backend_edges(backend), cvars)

# Create a system model
# =====
H_drift = 0.0
Hs_control = []
Hs_channels = []
As_static = []
for qubit in registers:
    Hj_drift, Hjs_control, Hjs_channel = H_rx(qubit)
    Ajs_static = A_decay(qubit)

    H_drift += Hj_drift
    Hs_control += Hjs_control
    Hs_channels += Hjs_channel
    As_static += Ajs_static

In [ ]:
np.max(As_static)

In [ ]:
np.max(H_xtalk)

# Solvers

In [ ]:
# Identity matrix
U0 = ps.qiskit_identity_operator(len(registers))
C0 = qk.quantum_info.SuperOp(U0)

# Get the carriers
registered_carriers = {c: carriers[c] for c in Hs_channels}
cutoff_freq = 2 * np.mean(list(registered_carriers.values()))

In [ ]:
# NOTE: Converting to a rotating frame leads to factor of 1/2 on controls.
solver= qk_d.Solver(
    static_hamiltonian=H_drift.data + H_xtalk.data,
    hamiltonian_operators=[2*H for H in Hs_control],
    hamiltonian_channels=Hs_channels,
    static_dissipators=As_static,
    rotating_frame=H_drift,
    rwa_cutoff_freq=None,
    channel_carrier_freqs=registered_carriers,
    dt=dt
)

# NOTE: Converting to a rotating frame leads to factor of 1/2 on controls.
solver0 = qk_d.Solver(
    static_hamiltonian=H_drift.data,
    hamiltonian_operators=[2*H for H in Hs_control],
    hamiltonian_channels=Hs_channels,
    # static_dissipators=As_static,
    rotating_frame=H_drift,
    rwa_cutoff_freq=None,
    channel_carrier_freqs=registered_carriers,
    dt=dt
)

# Schedules

In [ ]:
def build_pulse_schedule(list_of_moments, gateset, name, backend):
    """
    Build a pulse schedule from a circuit and a gateset.

    The circuit is specified as a list of moments, and a moment
    is a dictionary of {qubit: gate} pairs.
    """
    with qk.pulse.build(name=f"{name}", backend=backend) as schedule:
        for moment in list_of_moments:
            for qubit, gate_label in moment.items():
                channel = qk.pulse.DriveChannel(qubit)
                if "Y" in gate_label:
                    qk.pulse.shift_phase(np.pi / 2, channel)
                qk.pulse.play(gateset[gate_label], channel)
    return schedule

In [ ]:
layers = range(1, 10)

robust_schedules = []
for n in layers:
    moments = [{0: "X1", 1: "X2"}]
    moments += moments * (2 * (n - 1))
    sched = build_pulse_schedule(moments, x_gates, f"robust-{len(moments)}", backend)
    robust_schedules.append(sched)


default_schedules = []
for n in layers:
    moments = [{0: "X1", 1: "X2"}] 
    moments += moments * (2 * (n - 1))
    sched = build_pulse_schedule(
        moments, default_gates, f"default-{len(moments)}", backend
    )
    default_schedules.append(sched)

sched_keys = ["robust", "default"]

You need a list of durations corresponding to the accumulated gates.

In [ ]:
t_eval = [duration * dt * (2 * m - 1) for m in layers]

# NOTE: sometimes there is a leak and you need to reconstruct the solver

solver.model.evaluation_mode = 'sparse_vectorized'
# solver.model.evaluation_mode = 'sparse'

results = solver.solve(
    t_span=[0., max(t_eval)], 
    t_eval=t_eval, 
    # y0=U0, 
    y0=C0,
    signals=robust_schedules + default_schedules,
    atol=1e-12,
    rtol=1e-12,
    method='jax_odeint'
)

In [ ]:
solver0.model.evaluation_mode = 'sparse'
results0 = solver0.solve(
    t_span=[0., max(t_eval)], 
    t_eval=t_eval, 
    y0=U0, 
    signals=robust_schedules + default_schedules,
    atol=1e-12,
    rtol=1e-12,
    method='jax_odeint'
)

In [ ]:
x = [2 * i - 1 for i in layers]

In [ ]:
fidelity = []
for n, sol in enumerate(results):
    fidelity += [qk.quantum_info.process_fidelity(
        sol.y[n % len(t_eval)], ps.from_label("XX")
    )]

In [ ]:
fidelity0 = []
for n, sol in enumerate(results0):
    fidelity0 += [qk.quantum_info.process_fidelity(
        sol.y[n % len(t_eval)], ps.from_label("XX")
    )]

In [ ]:
fig, ax = plt.subplots()
ax.set_title("Crosstalk")
f = 1 - np.reshape(fidelity, (2, len(layers)))
for k, y in zip(sched_keys, f):
    ax.plot(x, y, marker='o', label=k)
# ax.set_yscale("log")
ax.set_xticks(x)
ax.legend()

In [ ]:
fig, axes = plt.subplots(2)
f0 = 1 - np.reshape(fidelity0, (2, len(layers)))

for ax in axes:
    for k, y in zip(sched_keys, f0):
        ax.plot(x, y, marker='o', label=k)
    ax.set_xticks(x)
    ax.legend()
axes[0].set_title("No crosstalk")
axes[1].set_yscale("log")


In [ ]:
fig, ax = plt.subplots()
f = 1 - np.reshape(fidelity, (2, len(layers)))
f0 = 1 - np.reshape(fidelity0, (2, len(layers)))
for k, y in zip(sched_keys, f / f0):
    ax.plot(x, y, marker='o', label=k)
ax.set_yscale("log")
ax.set_xticks(x)
ax.legend()